# Production pipeline (minimal)



In [19]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np

if not hasattr(np, "NINF"):
    np.NINF = -np.inf
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
    TargetEncoder,
)
from sklearn.ensemble import RandomForestClassifier
from mlxtend.feature_selection import SequentialFeatureSelector

import mlflow
import mlflow.sklearn

print("Imports OK")


Imports OK


In [20]:
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "clean_dataset.pkl"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)

df = pd.read_pickle(DATA_PATH)
print("df shape:", df.shape)
df.head()


PROJECT_ROOT = C:\Users\79022\Desktop\iis\iis
DATA_PATH = C:\Users\79022\Desktop\iis\iis\data\clean_dataset.pkl
df shape: (2000, 21)


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [21]:
TARGET_COL = "price_range"
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)


X_train: (1500, 20) X_test: (500, 20)


In [22]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("numeric_features:", len(numeric_features))
print("categorical_features:", len(categorical_features))


numeric_features: 20
categorical_features: 0


In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", TargetEncoder(target_type="multiclass"), categorical_features),
    ],
    remainder="drop",
)


In [24]:
# Feature engineering (sklearn): poly + bins
poly_features = ["ram", "battery_power"]
bin_features = ["px_height", "px_width", "int_memory"]

base_numeric_features = [
    col for col in numeric_features if col not in poly_features + bin_features
]

fe_preprocessor = ColumnTransformer(
    transformers=[
        ("num_base", StandardScaler(), base_numeric_features),
        (
            "poly",
            Pipeline(
                steps=[
                    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                    ("scale", StandardScaler()),
                ]
            ),
            poly_features,
        ),
        (
            "bins",
            Pipeline(
                steps=[("kbins", KBinsDiscretizer(n_bins=4, encode="onehot-dense"))]
            ),
            bin_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)


In [ ]:
ARTIFACTS_DIR = PROJECT_ROOT / "research" / "artifacts"

selected_feature_idx_path = ARTIFACTS_DIR / "selected_feature_indices.txt"
selected_feature_names_path = ARTIFACTS_DIR / "selected_feature_names.txt"

selected_feature_idx = [
    int(x.strip())
    for x in selected_feature_idx_path.read_text().splitlines()
    if x.strip()
]
selected_feature_names = [
    x.strip() for x in selected_feature_names_path.read_text().splitlines() if x.strip()
]

print("selected_feature_idx count:", len(selected_feature_idx))


selected_feature_idx count: 16


In [26]:
# Production pipeline (FE -> selection -> model)
from sklearn.base import clone

selector = ColumnTransformer(
    [("select", "passthrough", selected_feature_idx)],
    remainder="drop",
    verbose_feature_names_out=False,
)

production_pipeline = Pipeline(
    steps=[
        ("transform", clone(fe_preprocessor)),
        ("select", selector),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)),
    ]
)

X_full = X.copy()
y_full = y.copy()
production_pipeline.fit(X_full, y_full)
print("Production pipeline trained")


Production pipeline trained


In [27]:
# Save artifacts (feature names)
ARTIFACTS_DIR = PROJECT_ROOT / "research" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

production_feature_names_path = ARTIFACTS_DIR / "production_feature_names.txt"

X_full_fe = fe_preprocessor.fit_transform(X_full, y_full)
feature_names_full = fe_preprocessor.get_feature_names_out()
X_full_fe_df = pd.DataFrame(X_full_fe, columns=feature_names_full, index=X_full.index)
X_full_selected_df = X_full_fe_df.iloc[:, selected_feature_idx].copy()

with open(production_feature_names_path, "w", encoding="utf-8") as f:
    for col in X_full_selected_df.columns:
        f.write(f"{col}\n")

selected_feature_idx_path = ARTIFACTS_DIR / "selected_feature_indices.txt"
selected_feature_names_path = ARTIFACTS_DIR / "selected_feature_names.txt"

with open(selected_feature_idx_path, "w", encoding="utf-8") as f:
    for idx in selected_feature_idx:
        f.write(f"{idx}\n")

with open(selected_feature_names_path, "w", encoding="utf-8") as f:
    for name in selected_feature_names:
        f.write(f"{name}\n")


In [28]:
# Log production model to MLflow
from mlflow.models import infer_signature

production_input_example = X_full.head(5)
production_signature = infer_signature(
    model_input=X_full.head(5),
    model_output=production_pipeline.predict(X_full.head(5)),
)

RUN_NAME = "production_model_pipeline_minimal"
with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params(
        {
            "model_type": "Pipeline(RandomForestClassifier)",
            "stage": "production",
            "trained_on_full_dataset": True,
            "n_estimators": 100,
            "random_state": 42,
            "selected_features_count": len(selected_feature_idx),
        }
    )

    mlflow.log_artifact(str(production_feature_names_path))
    mlflow.log_artifact(str(selected_feature_idx_path))
    mlflow.log_artifact(str(selected_feature_names_path))
    mlflow.log_artifact(str(PROJECT_ROOT / "requirements.txt"))

    mlflow.sklearn.log_model(
        sk_model=production_pipeline,
        artifact_path="model",
        signature=production_signature,
        input_example=production_input_example,
        registered_model_name="mobile_price_classifier",
    )

    production_run_id = run.info.run_id

print("Production run logged to MLflow")
print("Run ID:", production_run_id)


Registered model 'mobile_price_classifier' already exists. Creating a new version of this model...
Created version '5' of model 'mobile_price_classifier'.


Production run logged to MLflow
Run ID: 8165633fb74d422da075b341cd030f41


In [29]:
# Set Production tag
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_name = "mobile_price_classifier"
latest_versions = client.search_model_versions(f"name='{model_name}'")
latest_version = max(latest_versions, key=lambda mv: int(mv.version))

client.set_model_version_tag(
    name=model_name, version=latest_version.version, key="Production", value="true"
)

print("Production set for version:", latest_version.version)


Production set for version: 5
